## SCOS Migration Output
**Source File**: `03_aggregations_pivot_windows_scos.ipynb`  
**Migrated on**: 2026-04-28  
**Conversion Tool**: Snowpark Connect (SCOS)

### Changes Applied
- SparkSession creation replaced with `snowpark_connect.init_spark_session()`
- PySpark imports annotated with SCOS compatibility comments
- Incompatible patterns flagged with `# SCOS-EWI:` markers

### Known Limitations
- Review `# SCOS-EWI:` markers for manual conversion required


# 03 — Aggregations, Pivot Tables, and Window Functions

**Tasty Bytes Consulting** — financial portfolio analytics demo.

Demonstrates advanced PySpark aggregation patterns over real financial concepts:

- `groupBy` with multiple aggregation functions (trade count, buy/sell notional, distinct assets)
- `pivot` to reshape portfolio × asset class from long → wide
- Window functions: `row_number`, `rank`, `dense_rank`
- Running totals using `rowsBetween(Window.unboundedPreceding, Window.currentRow)`
- `lag` and `lead` for month-over-month price comparison
- `collect_set` to gather traded assets per portfolio
- `rollup` for strategy → asset class → transaction type subtotals

In [ ]:
import sys; sys.path.insert(0, '.')
from demo_data import load_all, COMPANY_NAME

# SCOS: Replaced local SparkSession creation with snowpark_connect.init_spark_session().
# demo_data.create_spark_session() used SparkSession.builder.master('local[*]') which
# is not used in SCOS — the session is managed by the platform.
from snowflake import snowpark_connect
spark = snowpark_connect.init_spark_session()

# SCOS: Removed spark.sparkContext.setLogLevel('WARN') — SparkContext (RDD-level API)
# is not accessible in SCOS. Log level is managed by the Snowflake platform.
data = load_all(spark)

transactions_df = data["transactions"]
portfolios_df   = data["portfolios"]
assets_df       = data["assets"]
prices_df       = data["prices"]

# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql import functions as F
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql.window import Window

print(f"Spark version: {spark.version}")
print(f"Company: {COMPANY_NAME}")
print(f"Transactions: {transactions_df.count()} rows")
transactions_df.printSchema()


## 1. `groupBy` aggregations — portfolio × transaction type

Aggregate `fct_transactions` by `portfolio_id` and `txn_type` to summarise
trade activity: count, total buy/sell notional value (USD), distinct assets
traded, and average trade size.  Join `dim_portfolios` for the portfolio name.

In [ ]:
portfolio_txn_summary = (
    transactions_df
    .groupBy("portfolio_id", "txn_type")
    .agg(
        F.count("*").alias("trade_count"),
        F.round(
            F.sum(
                F.when(F.col("txn_type") == "buy",
                       F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd"))
                 .otherwise(0.0)
            ), 2
        ).alias("total_buy_usd"),
        F.round(
            F.sum(
                F.when(F.col("txn_type") == "sell",
                       F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd"))
                 .otherwise(0.0)
            ), 2
        ).alias("total_sell_usd"),
        F.countDistinct("asset_id").alias("distinct_assets"),
        F.round(
            F.avg(F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd")), 2
        ).alias("avg_trade_size_usd"),
    )
)

portfolio_summary = (
    portfolio_txn_summary
    .join(portfolios_df.select("portfolio_id", "portfolio_name"), "portfolio_id", "left")
    .select(
        "portfolio_id", "portfolio_name", "txn_type",
        "trade_count", "total_buy_usd", "total_sell_usd",
        "distinct_assets", "avg_trade_size_usd",
    )
    .orderBy("portfolio_id", "txn_type")
)

print("=== Portfolio x Transaction Type Summary ===")
portfolio_summary.show(truncate=False)

## 2. `pivot` — portfolio notional value by asset class

Reshape `fct_transactions` from long (one row per trade) to wide (one column
per asset class) showing total notional USD (`quantity × price × fx_rate_to_usd`).
Asset classes: `equity`, `fixed_income`, `alternative`, `cash`.

In [ ]:
# Compute notional value and attach asset_class from dim_assets
txn_with_class = (
    transactions_df
    .withColumn("notional_usd",
                F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd"))
    .join(assets_df.select("asset_id", "asset_class"), "asset_id", "left")
)

# Pivot: portfolio_id → one column per asset_class
notional_pivot = (
    txn_with_class
    .groupBy("portfolio_id")
    .pivot("asset_class", ["equity", "fixed_income", "alternative", "cash"])
    .agg(F.round(F.sum("notional_usd"), 2))
    .fillna(0.0)
    .join(
        portfolios_df.select("portfolio_id", "portfolio_name", "strategy"),
        "portfolio_id", "left"
    )
    .select(
        "portfolio_id", "portfolio_name", "strategy",
        "equity", "fixed_income", "alternative", "cash",
    )
    .orderBy("portfolio_id")
)

print("=== Notional USD Pivot: Portfolio x Asset Class ===")
notional_pivot.show(truncate=False)

## 3. Window functions — asset performance ranking

From `fct_daily_prices`, compute each asset's cumulative return relative to its
first available month-end close price, then rank all assets within each
month-end snapshot using `rank()`, `dense_rank()`, and `row_number()`.

In [ ]:
# Cumulative return from first available close price per asset
asset_time_window = Window.partitionBy("asset_id").orderBy("price_date")

prices_with_perf = (
    prices_df
    .withColumn("first_close", F.first("close_price").over(asset_time_window))
    .withColumn(
        "pct_return",
        F.round(
            (F.col("close_price") - F.col("first_close"))
            / F.col("first_close") * 100,
            2,
        ),
    )
)

# Rank assets by pct_return within each month-end snapshot
month_window = Window.partitionBy("price_date").orderBy(F.desc("pct_return"))

ranked_assets = (
    prices_with_perf
    .withColumn("rank",       F.rank().over(month_window))
    .withColumn("dense_rank", F.dense_rank().over(month_window))
    .withColumn("row_number", F.row_number().over(month_window))
    .join(assets_df.select("asset_id", "ticker", "asset_class"), "asset_id", "left")
    .select(
        "price_date", "ticker", "asset_class",
        "close_price", "pct_return", "rank", "dense_rank", "row_number",
    )
    .orderBy("price_date", "rank")
)

print("=== Asset Performance Ranking by Month-End ===")
ranked_assets.show(40, truncate=False)

print("=== Top Performer Each Month ===")
ranked_assets.filter(F.col("rank") == 1).show(truncate=False)

## 4. Running totals, `lag`, and `lead` — AAPL price history

For AAPL (AST-001) demonstrate:
- **Running total** of close prices using `rowsBetween(Window.unboundedPreceding, Window.currentRow)`
- **Cumulative return** from the first month-end observation
- **`lag`**: previous month's close price
- **`lead`**: next month's close price (look-ahead)

In [ ]:
aapl_window = Window.partitionBy("asset_id").orderBy("price_date")
rows_window  = aapl_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)

aapl_ts = (
    prices_df
    .filter(F.col("asset_id") == "AST-001")
    # Previous month close (lag)
    .withColumn("prev_close",       F.lag("close_price", 1).over(aapl_window))
    # Next month close (lead)
    .withColumn("next_month_close", F.lead("close_price", 1).over(aapl_window))
    # Month-over-month USD change
    .withColumn(
        "mom_change_usd",
        F.round(F.col("close_price") - F.col("prev_close"), 2),
    )
    # Month-over-month % change
    .withColumn(
        "mom_change_pct",
        F.when(
            F.col("prev_close").isNotNull(),
            F.round(
                (F.col("close_price") - F.col("prev_close"))
                / F.col("prev_close") * 100,
                2,
            ),
        ).otherwise(F.lit(None)),
    )
    # Running total of close prices (rowsBetween pattern)
    .withColumn(
        "running_total_usd",
        F.round(F.sum("close_price").over(rows_window), 2),
    )
    # Cumulative return from first observation
    .withColumn("first_close", F.first("close_price").over(aapl_window))
    .withColumn(
        "cumulative_return_pct",
        F.round(
            (F.col("close_price") - F.col("first_close"))
            / F.col("first_close") * 100,
            2,
        ),
    )
    .select(
        "price_date", "close_price", "prev_close", "next_month_close",
        "mom_change_usd", "mom_change_pct",
        "running_total_usd", "cumulative_return_pct",
    )
    .orderBy("price_date")
)

print("=== AAPL (AST-001): Running Total, Lag & Lead ===")
aapl_ts.show(truncate=False)

## 5. `collect_set` — traded assets per portfolio

Aggregate transactions to produce a per-portfolio profile: the set of distinct
asset IDs traded, first and last transaction dates, and total transaction count.

In [ ]:
portfolio_profile = (
    transactions_df
    .groupBy("portfolio_id")
    .agg(
        F.collect_set("asset_id").alias("traded_assets"),
        F.min("txn_date").alias("first_trade_date"),
        F.max("txn_date").alias("last_trade_date"),
        F.count("*").alias("total_transactions"),
    )
    .join(
        portfolios_df.select("portfolio_id", "portfolio_name", "strategy"),
        "portfolio_id", "left"
    )
    .select(
        "portfolio_id", "portfolio_name", "strategy",
        "traded_assets", "first_trade_date", "last_trade_date", "total_transactions",
    )
    .orderBy("portfolio_id")
)

print("=== Portfolio Trading Profile (collect_set) ===")
portfolio_profile.show(truncate=False)

## 6. `rollup` — multi-level subtotals

`rollup` generates subtotals at each level of a hierarchy plus a grand total.
Roll up `fct_transactions` by `strategy` → `asset_class` → `txn_type`,
joining `dim_portfolios` and `dim_assets` first to attach those dimensions.

In [ ]:
# Enrich transactions with strategy (from portfolios) and asset_class (from assets)
txn_enriched = (
    transactions_df
    .withColumn("notional_usd",
                F.col("quantity") * F.col("price") * F.col("fx_rate_to_usd"))
    .join(portfolios_df.select("portfolio_id", "strategy"), "portfolio_id", "left")
    .join(assets_df.select("asset_id", "asset_class"),       "asset_id",     "left")
)

rollup_df = (
    txn_enriched
    .rollup("strategy", "asset_class", "txn_type")
    .agg(
        F.round(F.sum("notional_usd"), 2).alias("total_notional_usd"),
        F.count("*").alias("trade_count"),
    )
    .withColumn(
        "level",
        F.when(F.col("strategy").isNull(),   "Grand Total")
         .when(F.col("asset_class").isNull(), "Strategy Subtotal")
         .when(F.col("txn_type").isNull(),    "Asset Class Subtotal")
         .otherwise("Detail"),
    )
    .orderBy(
        F.col("strategy").asc_nulls_last(),
        F.col("asset_class").asc_nulls_last(),
        F.col("txn_type").asc_nulls_last(),
    )
)

print("=== Rollup: Strategy → Asset Class → Transaction Type ===")
rollup_df.show(50, truncate=False)

In [ ]:
# SCOS: Removed spark.stop() — do not terminate the SCOS-managed session
# explicitly. The platform manages session lifecycle.
